# 06 — cGAN Inspection: EuroSAT · Seed 42

**Purpose:** Post-training inspection of the EuroSAT conditional WGAN-GP.  
This notebook verifies data integrity of the saved synthetic `.npz`, confirms model
checkpoint artefacts were written correctly, reviews the training progression through
saved epoch grids, assesses per-class sample quality, and provides a direct
real-vs-synthetic visual comparison.

> **Scope note:** This notebook is for generative model inspection only.
> Downstream classifier evaluation on synthetic data is reserved for the research phase
> (notebooks 08+) and must not be performed here.

---
## 0 · Configuration

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE          = '/path/to/thesis_project'
SYNTH_PATH    = f'{BASE}/data/synthetic/seed_42/cgan/eurosat/eurosat_synthetic.npz'
REAL_PATH     = f'{BASE}/data/processed/seed_42/eurosat_seed_42.npz'
MODEL_DIR     = f'{BASE}/models/cgan/eurosat/seed_42/'

# ── Dataset constants ─────────────────────────────────────────────────────────
CLASS_NAMES = [
    'AnnualCrop', 'Forest', 'HerbaceousVeg', 'Highway', 'Industrial',
    'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake'
]
N_CLASSES  = 10
IMAGE_SIZE = 64
CHANNELS   = 3

print('Configuration loaded.')
print(f'  Synthetic data : {SYNTH_PATH}')
print(f'  Real data      : {REAL_PATH}')
print(f'  Model dir      : {MODEL_DIR}')

---
## 1 · Synthetic Data Verification

Verify that the saved `.npz` is structurally correct before any qualitative inspection:
expected keys, shapes, dtype, and pixel range.  
Pixel range is checked for **all splits** — a range check on training data alone can
miss clipping or normalisation errors in the validation split.

In [ ]:
data = np.load(SYNTH_PATH)

print('── Keys ──────────────────────────────────')
print(list(data.keys()))

print('\n── Shapes ────────────────────────────────')
for k in data.keys():
    print(f'  {k}: {data[k].shape}')

print('\n── Dtypes ────────────────────────────────')
for k in data.keys():
    print(f'  {k}: {data[k].dtype}')

print('\n── Pixel range (ALL splits) ──────────────')
for split in ['X_train', 'X_val']:
    lo, hi = data[split].min(), data[split].max()
    status = '✓' if lo >= -1.01 and hi <= 1.01 else '✗  OUT OF RANGE'
    print(f'  {split}: [{lo:.4f}, {hi:.4f}]  {status}')

### 1.1 · Class Distribution

Check that synthetic samples are evenly distributed across all 10 classes in both splits.
Imbalance here would indicate a label conditioning failure during generation.

In [ ]:
for split_x, split_y in [('X_train', 'y_train'), ('X_val', 'y_val')]:
    y = data[split_y]
    unique, counts = np.unique(y, return_counts=True)
    total = len(y)

    print(f'── {split_y}  (n={total}) ──────────────────────')
    for cls_idx, cnt in zip(unique, counts):
        bar = '█' * int(30 * cnt / total)
        name = CLASS_NAMES[int(cls_idx)]
        print(f'  {int(cls_idx):>2d}  {name:<15}  {cnt:>5d}  {bar}')
    print()

---
## 2 · Model Checkpoint Verification

Confirm that generator and critic weight files were written to disk and are non-zero in size.

In [ ]:
print(f'Files in: {MODEL_DIR}\n')
all_files = []
for root, dirs, files in os.walk(MODEL_DIR):
    dirs.sort()
    for f in sorted(files):
        full = os.path.join(root, f)
        rel  = os.path.relpath(full, MODEL_DIR)
        size = os.path.getsize(full)
        flag = '✓' if size > 0 else '✗  EMPTY'
        print(f'  {rel:<55}  {size/1e6:>7.2f} MB  {flag}')
        all_files.append(f)

print(f'\nTotal files: {len(all_files)}')

---
## 3 · Training Progression

Visualise the generator output grids saved at evenly-spaced checkpoints during training.
These grids are the primary tool for spotting mode collapse, training instability, or
progressive quality improvement epoch-by-epoch.

Available checkpoint paths are detected automatically; if fewer than expected are found
the cell prints a warning rather than raising an error.

In [ ]:
final_dir = os.path.join(MODEL_DIR, 'final')

# ── Auto-detect available epoch grid images ────────────────────────────────────
available = []
if os.path.isdir(final_dir):
    for entry in sorted(os.listdir(final_dir)):
        entry_path = os.path.join(final_dir, entry)
        if os.path.isdir(entry_path) and entry.startswith('epoch_'):
            img_name = f'generated_{entry}.png'
            img_path = os.path.join(entry_path, img_name)
            if os.path.isfile(img_path):
                epoch_num = int(entry.split('_')[1])
                available.append((epoch_num, img_path))

if not available:
    print('WARNING: No epoch grid images found. Check that SaveModels callback'
          ' wrote grids to models/cgan/eurosat/seed_42/final/epoch_XXX/')
else:
    # Select up to 6 evenly-spaced checkpoints
    step = max(1, len(available) // 6)
    selected = available[::step][:6]
    n = len(selected)

    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]

    for ax, (ep, path) in zip(axes, selected):
        img = Image.open(path)
        ax.imshow(img)
        ax.set_title(f'Epoch {ep}', fontsize=10)
        ax.axis('off')

    plt.suptitle('EuroSAT cGAN — Training Progression', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig('eurosat_training_progression.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Showing {n} of {len(available)} available checkpoints.')

---
## 4 · Synthetic Sample Quality — Per-Class Grid

Display 5 randomly-sampled synthetic images per class from `X_train`.
Images are stored in the [−1, 1] range and rescaled to [0, 1] for display.

In [ ]:
X_syn = data['X_train']
y_syn = data['y_train']

SAMPLES_PER_CLASS = 5
rng = np.random.default_rng(seed=42)

fig, axes = plt.subplots(N_CLASSES, SAMPLES_PER_CLASS,
                         figsize=(SAMPLES_PER_CLASS * 2, N_CLASSES * 2))

for cls in range(N_CLASSES):
    cls_idx = np.where(y_syn == cls)[0]
    chosen  = rng.choice(cls_idx, size=min(SAMPLES_PER_CLASS, len(cls_idx)), replace=False)
    for j, i in enumerate(chosen):
        img = np.clip((X_syn[i] + 1) / 2, 0, 1)   # [-1,1] → [0,1]
        axes[cls, j].imshow(img)
        axes[cls, j].axis('off')
    axes[cls, 0].set_title(CLASS_NAMES[cls], fontsize=8, loc='left')

plt.suptitle('EuroSAT cGAN — Synthetic Samples (X_train)', fontsize=12)
plt.tight_layout()
plt.savefig('eurosat_synthetic_samples.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5 · Real vs. Synthetic Comparison

Side-by-side visual comparison of real (processed, [−1, 1]) and synthetic images for
each class.  This is the most direct qualitative check that the generator has learned
the correct class-conditional appearance — texture, colour palette, and spatial structure
should be broadly consistent between columns.

In [ ]:
real = np.load(REAL_PATH)
X_real = real['X_train']
y_real = real['y_train']

print(f'Real data loaded:  X_train {X_real.shape}, dtype {X_real.dtype}')
print(f'Pixel range (real X_train): [{X_real.min():.4f}, {X_real.max():.4f}]')

In [ ]:
COMPARE_COLS = 4    # images per split per class
rng = np.random.default_rng(seed=42)

fig, axes = plt.subplots(
    N_CLASSES, COMPARE_COLS * 2 + 1,
    figsize=((COMPARE_COLS * 2 + 1) * 1.6, N_CLASSES * 1.8)
)

for cls in range(N_CLASSES):
    col = 0

    # Real images
    real_idx = rng.choice(np.where(y_real == cls)[0], size=COMPARE_COLS, replace=False)
    for i in real_idx:
        img = np.clip((X_real[i] + 1) / 2, 0, 1)
        axes[cls, col].imshow(img)
        axes[cls, col].axis('off')
        if cls == 0:
            axes[cls, col].set_title('Real', fontsize=7)
        col += 1

    # Divider column
    axes[cls, col].axis('off')
    if cls == 0:
        axes[cls, col].set_title('|', fontsize=7)
    col += 1

    # Synthetic images
    syn_idx = rng.choice(np.where(y_syn == cls)[0], size=COMPARE_COLS, replace=False)
    for i in syn_idx:
        img = np.clip((X_syn[i] + 1) / 2, 0, 1)
        axes[cls, col].imshow(img)
        axes[cls, col].axis('off')
        if cls == 0:
            axes[cls, col].set_title('Synthetic', fontsize=7)
        col += 1

    # Row label
    axes[cls, 0].set_ylabel(CLASS_NAMES[cls], fontsize=7, rotation=0,
                            labelpad=55, va='center')

plt.suptitle('EuroSAT — Real vs. Synthetic (per class)', fontsize=12)
plt.tight_layout()
plt.savefig('eurosat_real_vs_synthetic.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6 · Summary

Use this cell to record observations from the inspection run.

| Check | Status | Notes |
|---|---|---|
| Keys / shapes | | |
| Dtype | | |
| Pixel range — X_train | | |
| Pixel range — X_val | | |
| Class balance — y_train | | |
| Class balance — y_val | | |
| Checkpoint files present | | |
| Training progression | | |
| Sample quality (visual) | | |
| Real vs. synthetic (visual) | | |

**Overall assessment:**  
*(fill in after running)*